In [85]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForQuestionAnswering

In [ ]:
device = "cpu" #Workaround for CUDA kernel errors


Phase1:"Using the BLIP-Base model as a baseline, I uploaded diverse single-frame inputs and varied prompts. However, the model failed to deliver stable or consistent predictions, highlighting a lack of spatial robustness in zero-shot staircase detection.


In [87]:
model_id = "Salesforce/blip-vqa-base"
processor = BlipProcessor.from_pretrained(model_id)
model = BlipForQuestionAnswering.from_pretrained(model_id).to(device)

Loading weights: 100%|██████████| 788/788 [00:00<00:00, 10386.35it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForQuestionAnswering LOAD REPORT from: Salesforce/blip-vqa-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_encoder.embeddings.position_ids      | UNEXPECTED |  | 
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [88]:
def recognize_stair(image_path=r"E:\winter\materials\up_stair.jpg"):
    try:
        image = Image.open(image_path).convert('RGB')
        prompt = "What is the direction of the stairs in the image? Answer with one word: 'up' or 'down' or 'none'. "
        inputs = processor(image, text=prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=20)
            answer = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

        return answer
    
    except FileNotFoundError:
        return "找不到图片文件"
    except Exception as e:
        return f"运行出错: {str(e)}"

In [89]:
result = recognize_stair()
print(f"检测结果 : {result}")

检测结果 : down


Phase2:Explored diverse methodologies to enhance predictive accuracy and robustness in staircase orientation perception.

In [90]:

#Lab1:Temporal Consistency Experiment
import time
def temporal_comparison(up_img_path, near_img_path):
    res_far = recognize_stair(r"E:\winter\materials\down_stair.jpg")
    print(f"远景识别结果: {res_far}")

    res_near = recognize_stair(r"E:\winter\materials\down_stair2.jpg")
    print(f"近景识别结果: {res_near}")

temporal_comparison(r"E:\winter\materials\down_stair.jpg", r"E:\winter\materials\down_stair2.jpg")



远景识别结果: down
近景识别结果: down


In [91]:
#Lab2:Model Scaling

#Memory Release
import gc

if 'model' in locals():
    del model
if 'processor' in locals():
    del processor

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()    

In [ ]:
model_id = "Salesforce/blip-vqa-capfilt-large"

device = "cpu" #Workaround for CUDA kernel errors

from transformers import BlipProcessor, BlipForQuestionAnswering

processor = BlipProcessor.from_pretrained(model_id)
model = BlipForQuestionAnswering.from_pretrained(model_id).to(device)

In [14]:
from PIL import Image

img_path = r"E:\winter\materials\up_stair.jpg"
raw_image = Image.open(img_path).convert('RGB')
prompt = "Is the camera located at the bottom of the stairs or at the top? "

inputs = processor(raw_image, prompt, return_tensors="pt").to(device)
out = model.generate(**inputs, max_new_tokens=20)
answer = processor.decode(out[0], skip_special_tokens=True)

print(f"检测结果：{answer}")

检测结果：bottom


Phase3:Based on lab1 and lab2, I extended my investigation into Sequential Imagery Experiments, stimulating the active perception of a robot by analyzing image sequences captured during continuous forward movement.

In [30]:
#Lab3:Sequential Perception
import os
from PIL import Image

folder_path = r"E:\winter\materials\up_stair"
image_files = sorted([f for f in os.listdir(folder_path) if f.endswith(('.jpg'))])

results = []
for img_name in image_files:
    img_path = os.path.join(folder_path, img_name)
    raw_image = Image.open(img_path).convert('RGB')

    prompt = "Imagine you take one step forward. Will your altitude increase or decrease?" \
    
    
    inputs = processor(raw_image, prompt, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=10)
    answer = processor.decode(out[0], skip_special_tokens=True)
    results.append((img_name, answer))

    results.append((img_name, answer))
    print(f"{img_name} 预测结果: {answer}")

step1.jpg 预测结果: higher
step2.jpg 预测结果: higher
step3.jpg 预测结果: higher
